In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# If your folder is named "Questionnaire" instead of
# "Questionnnair", change only this folder name.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_I.xpt"
csv_file = DATA_DIR / "ACQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 ACQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD011A",
    "ACD011B",
    "ACD011C",
    "ACD040",
    "ACD110"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ACQ_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8991:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,991 rows x 6 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached ACQ_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
# All populated values are whole-number IDs/codes.
#
#
# Therefore all 6 variables can safely become nullable Int64.
#
#
# IMPORTANT:
#
# This applies ONLY to ACQ_I.
#
# It does NOT change your broader NHANES fidelity rule:
#
#     Dietary decimals
#     Examination continuous measurements
#     Laboratory continuous measurements
#     survey/sample weights
#
# must remain FLOAT / AS IS whenever they genuinely
# contain continuous decimal values.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 6:

    raise ValueError(
        "Expected exactly 6 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# IMPORTANT:
#
# Attached ACQ_I contains NONE.
#
# Therefore NO tiny -> zero replacement should be done.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 6 ACQ_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in ACQ_I."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zeros "
        "found in ACQ_I."
    )


print(
    "PASS: ACQ_I contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# No tiny-zero corruption exists in this attachment.
#
# Therefore the original XPT is directly the
# numeric fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero restoration required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a diagnostic comparison.
#
# NO dataframe values are rounded.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in ACQ_I.\n\n"
        "They will NOT be rounded:\n"
        + str(
            detected_decimal_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in ACQ_I."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 9. VALIDATE ALL NON-SEQN FIELDS ARE WHOLE-NUMBER
# ============================================================

for col in [
    "ACD011A",
    "ACD011B",
    "ACD011C",
    "ACD040",
    "ACD110"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All ACQ response variables "
    "contain only whole-number codes."
)


# ============================================================
# 10. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO round() during conversion.
#
# All six variables have already been proven whole-number.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 6 ACQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - ACD011A
# ============================================================

#
# Language spoken at home for:
#
#     non-Hispanic white
#     non-Hispanic black
#     other race / multiracial
#
#
# 1  = English
# 99 = Don't know
#
# CDC:
#
#     English    = 4,963
#     Don't know =     1
#     Missing    = 4,027
#

check_frequency(
    "ACD011A",
    {
        1: 4963,
        99: 1
    },
    4027
)


# ============================================================
# 13. CDC CHECK - ACD011B
# ============================================================

#
# 8 = Spanish
#
# CDC:
#
#     Spanish = 16
#     Missing = 8,975
#

check_frequency(
    "ACD011B",
    {
        8: 16
    },
    8975
)


# ============================================================
# 14. CDC CHECK - ACD011C
# ============================================================

#
# 9 = Other language
#
# CDC:
#
#     Other   = 244
#     Missing = 8,747
#

check_frequency(
    "ACD011C",
    {
        9: 244
    },
    8747
)


# ============================================================
# 15. CDC CHECK - ACD040
# ============================================================

#
# Language spoken at home - Hispanic participants:
#
#     1 = Only Spanish
#     2 = More Spanish than English
#     3 = Both equally
#     4 = More English than Spanish
#     5 = Only English
#     7 = Refused
#     9 = Don't know
#
#
# CDC attachment frequency:
#
#     1 = 767
#     2 = 438
#     3 = 533
#     4 = 554
#     5 = 588
#     7 =   0
#     9 =   2
#
# Missing = 6,109
#

check_frequency(
    "ACD040",
    {
        1: 767,
        2: 438,
        3: 533,
        4: 554,
        5: 588,
        9: 2
    },
    6109
)


# ============================================================
# 16. CDC CHECK - ACD110
# ============================================================

#
# Language spoken at home - non-Hispanic Asian:
#
#     1 = Only non-English
#     2 = More non-English than English
#     3 = Both equally
#     4 = More English than non-English
#     5 = Only English
#     7 = Refused
#     9 = Don't know
#
#
# CDC:
#
#     1 = 412
#     2 = 101
#     3 =  90
#     4 = 137
#     5 = 241
#     7 =   0
#     9 =   3
#
# Missing = 8,007
#

check_frequency(
    "ACD110",
    {
        1: 412,
        2: 101,
        3: 90,
        4: 137,
        5: 241,
        9: 3
    },
    8007
)


# ============================================================
# 17. NONMISSING TOTAL CHECKS
# ============================================================

EXPECTED_NONMISSING = {
    "ACD011A": 4964,
    "ACD011B": 16,
    "ACD011C": 244,
    "ACD040": 2882,
    "ACD110": 984
}


print(
    "\n--- NONMISSING TOTAL CHECK ---"
)


for col, expected_count in EXPECTED_NONMISSING.items():

    actual_count = int(
        df[col]
        .notna()
        .sum()
    )


    print(
        f"{col}:",
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} nonmissing values, "
            f"found {actual_count:,}."
        )


print(
    "PASS: All variable nonmissing counts validated."
)


# ============================================================
# 18. VALID RESPONSE-CODE CHECK
# ============================================================

#
# This detects unexpected numeric codes.
#

VALID_CODES = {
    "ACD011A": {1, 99},
    "ACD011B": {8},
    "ACD011C": {9},
    "ACD040": {1, 2, 3, 4, 5, 9},
    "ACD110": {1, 2, 3, 4, 5, 9}
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    print(
        f"{col}:",
        sorted(observed_codes)
    )


    unexpected = (
        observed_codes
        -
        allowed_codes
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected response codes."
)


# ============================================================
# 19. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Original XPT had:
#
#     tiny values = 0
#     ordinary zeros = 0
#
# Cleaning must therefore NOT create numeric zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 20. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ACQ_I required NO numeric repair.
#
# Therefore all six variables must remain numerically
# identical to the source XPT.
#
# Changing:
#
#     83732.0 -> 83732
#
# affects CSV formatting only, not the numeric value.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 6 variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# ACQ_I contains NO genuine decimals.
#
# Still use the same safe export standard:
#
#     NO round()
#     NO float_format=
#
#
# This keeps the program consistent with Dietary,
# Laboratory, Examination and Questionnaire files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 22. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 23. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 6 ACQ_I variables."
)


# ============================================================
# 24. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 25. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 26. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 27. RAW CSV .0 CHECK
# ============================================================

#
# Unlike mixed files such as:
#
#     GLU_I
#     TRIGLY_I
#     BMX_I
#     Dietary files
#
# ACQ_I contains NO genuine decimal variables.
#
# Therefore checking all six columns for ".0" is safe.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "any ACQ_I variable."
)


# ============================================================
# 28. FINAL CSV FREQUENCY CHECK
# ============================================================

#
# Re-read numeric values and make sure the output file
# still contains the exact response frequencies.
#

final_frequency_checks = {
    "ACD011A": {
        1: 4963,
        99: 1
    },

    "ACD011B": {
        8: 16
    },

    "ACD011C": {
        9: 244
    },

    "ACD040": {
        1: 767,
        2: 438,
        3: 533,
        4: 554,
        5: 588,
        9: 2
    },

    "ACD110": {
        1: 412,
        2: 101,
        3: 90,
        4: 137,
        5: 241,
        9: 3
    }
}


print(
    "\n--- FINAL CSV FREQUENCY CHECK ---"
)


for col, expected_freq in final_frequency_checks.items():

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        numeric_values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    print(
        f"{col}:",
        actual_freq
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final CSV {col} frequency "
            "distribution changed."
        )


print(
    "PASS: Final response frequencies preserved."
)


# ============================================================
# 29. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ACQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Acculturation"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "ACD011A nonmissing:",
    f"{int(df['ACD011A'].notna().sum()):,}"
)


print(
    "ACD011B nonmissing:",
    f"{int(df['ACD011B'].notna().sum()):,}"
)


print(
    "ACD011C nonmissing:",
    f"{int(df['ACD011C'].notna().sum()):,}"
)


print(
    "ACD040 nonmissing:",
    f"{int(df['ACD040'].notna().sum()):,}"
)


print(
    "ACD110 nonmissing:",
    f"{int(df['ACD110'].notna().sum()):,}"
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "ACQ_I contains no genuine decimal variables."
)


print(
    "All six variables can safely be exported "
    "as whole-number IDs/codes."
)


print(
    "The broader NHANES rule remains unchanged: "
    "genuine Dietary and other continuous decimal "
    "variables stay exactly AS IS."
)

NHANES 2015-2016 ACQ_I XPT -> CSV
Rows: 8,991
Columns: 6
PASS: Structure = 8,991 rows x 6 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'ACD011A', 'ACD011B', 'ACD011C', 'ACD040', 'ACD110']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: ACQ_I contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero restoration required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
ACD011A: 0 fractional observations
ACD011B: 0 fractional observations
ACD011C: 0 fractional observations
ACD040: 0 fractional observations
ACD110: 0 fractional observations

PASS: No genuine decimal variables found in ACQ_I.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.
PASS: All ACQ response variables contain only whole-number codes.

PASS: All 6 ACQ_I variables converted to nulla